In [65]:
%load_ext autoreload
%autoreload 2

Failed to reload module 'broharness.data_model' from file 'D:\study-on-agent\src\broharness\data_model.py'
Traceback (most recent call last):
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 455, in update_generic
    update(a, b)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 407, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 455, in update_generic
    update(a, b)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 367, in update_funct

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of broharness.data_model failed: Traceback (most recent call last):
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 455, in update_generic
    update(a, b)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 407, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 455, in update_generic
    update(a, b)
  File "d:\study-on-agent\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 367, in update_function
    setattr(old, name, getattr(new, name))
ValueError: __i

In [66]:
from broharness.llms.bedrock import bedrock, UserMessage, AIMessage, SystemMessage
from broharness.flows.skill_call import SkillCall
from broharness.flows.tool_call import ToolCall
from broharness.flows.tool_use import ToolUse
from broharness.flows.ask_user_question import AskUserQuestion
from broharness.flows.fail_recovery import FailRecovery
from broharness.flows.answer import Answer
from broflow import BaseTask, TaskRegistry, Flow
from pathlib import Path
import yaml
import sys
import subprocess
from broskill import SkillControl, ToolControl
from functools import partial
from broharness.toolblock import (
    tool_to_yaml, 
    load_skill_tool, 
    load_skill_extension_tool, 
    load_tool_tool, 
    ask_user_question_tool
)
from broharness.codeblock import parse_json_codeblock
from broharness.data_model import Process, State, LLMUse


ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
sc.list_skills()
tc = ToolControl(sc)

In [67]:
from broskill.processing.tool import to_args

In [68]:
tc.load_tool('read-file', 'scripts/read_file.py')

Tool(name='read_file', description="Read exactly one file's content, found via a glob pattern that must match a single file.", args=[Arg(name='pattern', type='string', description="Glob pattern, relative to the project root, that matches exactly one file (e.g. 'skills/tell-joke/references/dad-joke.md').", required=True)], path=WindowsPath('D:/study-on-agent/skills/read-file/scripts/read_file.py'))

In [69]:
TOOLS = dict(
    load_skill=sc.load_skill,
    load_skill_extension=sc.load_skill_extension,
    load_tool=tc.load_tool,
)

In [70]:
skill_call = SkillCall(name='skill-call', llm=bedrock, system_prompt='')
tool_call = ToolCall(name='tool-call', llm=bedrock, system_prompt='')
tool_use = ToolUse(name='tool-use', llm=bedrock, system_prompt='')
ask_user_question = AskUserQuestion(name='ask-user-question', llm=bedrock, system_prompt='')
fail_recovery = FailRecovery(name='fail-recovery', llm=bedrock, system_prompt='')
answer = Answer(name='answer', llm=bedrock, system_prompt='')

In [71]:
registry = TaskRegistry()
registry.register(Process.SKILL_CALL, skill_call)
registry.register(Process.TOOL_CALL, tool_call)
registry.register(Process.TOOL_USE, tool_use)
registry.register(Process.ASK_USER_QUESTION, ask_user_question)
registry.register(Process.FAIL_RECOVERY, fail_recovery)
registry.register(Process.ANSWER, answer)

flow = Flow(registry)

In [72]:
system_prompt = "You're Andy who is the best bro in the world. Always response in bro-tone with chill and mellow manner."

In [73]:
# this trigger load_skill with skill_name='tell-jokes'
content = "tell me some jokes."
# this trigger ask_user_question
# content = "What's the capital of France?"
# this trigger nothing
# content = "1+1 is?"
messages = [UserMessage(content)]
state = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages,
    session_messages=messages.copy(),
    system_prompt=system_prompt,
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages.copy()
)

_ = flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill', 'input': {'skill_name': 'tell-joke'}}]
load_skill passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'ask_user_question', 'input': {'question': 'What kind of joke would you like to hear?'}}]
ask_user_question passed
D:\study-on-agent\src\broharness\flows\ask_user_question.py
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill_extension', 'input': {'skill_name': 'tell-joke', 'path': 'references/dad-joke.md'}}, {'name': 'load_skill_extension', 'input': {'skill_name': 'tell-joke', 'path': 'references/pun-joke.md'}}]
load_skill_extension passed
load_skill_extension passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\answer.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'

In [74]:
state.messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Why did Batman start bringing a ladder to his fights with the Joker?\n\nBecause he heard the Joker was on a higher plane of villainy!'}]}]

In [75]:
state.session_messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'What kind of joke would you like to hear?'}]},
 {'role': 'user', 'content': [{'text': 'dad jokes and puns mixed together.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Okay, great! And what topic should the jokes be about?'}]},
 {'role': 'user', 'content': [{'text': 'Can it be about Batman and Joker?'}]},
 {'role': 'assistant',
  'content': [{'text': 'Why did Batman start bringing a ladder to his fights with the Joker?\n\nBecause he heard the Joker was on a higher plane of villainy!'}]}]

In [76]:
state.registered_skills

{'tell-joke': '# Tell Joke\n\n## Instructions\n\n- Ask the user which kind of joke they\'d like: dad jokes, puns, knock-knock\n  jokes, one-liners, riddles, or a mix of any of these.\n- Also ask what topic the joke should be about (e.g. work, food, animals) -- as its\n  own separate `ask_user_question` call, not bundled into the joke-type question.\n  No topic preference is a fine answer too; don\'t force a pick.\n- If they already said the type and/or topic in their request, don\'t ask again for\n  whichever part they already gave.\n- If their answer is unclear, ask a short clarifying question. Call `ask_user_question`.\n- Tell exactly one joke at a time, in your own words. Don\'t paste a reference file\'s\n  contents back verbatim -- use it as a style guide to write a fresh joke on the\n  requested topic, not a script to copy.\n- If more than one reference is loaded (a "both"/"all"/multi-type request), that\'s\n  still one joke, not one per style -- fuse every loaded style\'s charact

In [82]:
print('\n\n'.join(state.extension_skills.values()))

# Dad jokes

Dad jokes are short, deadpan one-liners built on the most literal, groan-worthy
reading of a common phrase or word. The humor comes from how obvious and corny
the pun is -- being "so bad it's good" is the point, not a flaw.

## Characteristics

- Plays on a double meaning or overly literal reading of an everyday phrase or idiom.
- Delivered deadpan, straight-faced -- no self-aware "get it?" follow-up.
- Family-friendly, no edge or innuendo.
- Short: one or two sentences, the setup and punchline folded together.
- Usually built around a mundane, everyday topic (food, chores, animals, objects,
  work) -- swap in whatever topic the user asked for.

Use this as a style guide to write a fresh, original dad joke on the requested
topic -- not a script to copy from verbatim.
# Puns

Puns rely on wordplay -- a word or phrase that sounds like another, or carries
two valid meanings, used so the double meaning lands as a surprise at the end.

## Characteristics

- Built on a homophone

In [78]:
state.session_tools

{'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x00000267CCCB2EA0>>,
 'load_skill_extension': <bound method SkillControl.load_skill_extension of <broskill.processing.skill.SkillControl object at 0x00000267CCCB2EA0>>,
 'load_tool': <bound method ToolControl.load_tool of <broskill.processing.tool.ToolControl object at 0x00000267CD005250>>}

In [79]:
state.error_message

''

In [80]:
state.debug

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "tell-joke" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1052, 'outputTokens': 47, 'totalTokens': 1099}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "ask_user_question",\n      "input": {\n        "question": "What kind of joke would you like to hear?"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1681, 'outputTokens': 63, 'totalTokens': 1744}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/dad-joke.md" } },\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/pun-joke.md" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1708, 'outputTokens': 102, 'totalTokens

In [86]:
state.usage

{'XXX_CALL': {'google.gemma-3-12b-it': {'input_tokens': 6521,
   'output_tokens': 314,
   'call_count': 4}},
 'ANSWER': {'google.gemma-3-12b-it': {'input_tokens': 2119,
   'output_tokens': 44,
   'call_count': 2}}}

In [97]:
def cal_usage(input, output, input_price, output_price, currency=1, session=1):
    mil = 1_000_000
    inputPrice = (input/mil)*input_price*currency*session
    outputPrice = (output/mil)*output_price*currency*session
    return inputPrice, outputPrice, inputPrice+outputPrice

In [98]:
cal_usage(6521+2119, 314+44, 0.09, 0.29)

(0.0007775999999999999, 0.00010381999999999999, 0.00088142)

In [101]:
cal_usage(6521+2119, 314+44, 0.09, 0.29, 34, 100)

(2.6438399999999995, 0.35298799999999997, 2.9968279999999994)

In [85]:
m_usage = {}
for m_type, m in state.usage.items():
    for _m, u in m.items():
        # if _m not in m_usage:
        #     m_usage[_m] = {"input_tokens": 0, "output_tokens": 0, "call_count": 0}
        print(f'{m_type} - {_m}: {u}')

XXX_CALL - google.gemma-3-12b-it: {'input_tokens': 6521, 'output_tokens': 314, 'call_count': 4}
ANSWER - google.gemma-3-12b-it: {'input_tokens': 2119, 'output_tokens': 44, 'call_count': 2}
